In [1]:
from abc import ABC, abstractmethod  # Importa ABC y abstractmethod para definir clases y métodos abstractos
from datetime import datetime  # Importa datetime para registrar fecha y hora en objetos y registros
import logging  # Importa logging para escribir eventos y errores en un archivo de texto


def configurar_bitacora_fj(archivo_log: str = "bitacora_software_fj.log") -> logging.Logger:
    """Crea y configura el logger que guarda eventos y errores en un archivo .log."""
    logger_fj = logging.getLogger("BitacoraFJ")  # Obtiene o crea el logger con nombre propio
    logger_fj.setLevel(logging.INFO)  # Registra mensajes informativos (INFO) y de error (ERROR)
    if not logger_fj.handlers:  # Evita agregar manejadores repetidos si se ejecuta la celda más de una vez
        manejador_txt = logging.FileHandler(archivo_log, mode="w", encoding="utf-8")  # Abre el archivo .log en UTF-8
        formato_linea = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")  # Define fecha, nivel y mensaje
        manejador_txt.setFormatter(formato_linea)  # Asigna el formato al manejador de archivo
        logger_fj.addHandler(manejador_txt)  # Vincula el manejador al logger
    return logger_fj  # Retorna el logger configurado


bitacora = configurar_bitacora_fj()  # Inicializa la bitácora global del sistema
print("Paso 1 verificado: bitácora 'bitacora_software_fj.log' configurada.")  # Confirma ejecución de la celda

Paso 1 verificado: bitácora 'bitacora_software_fj.log' configurada.


In [2]:
class BaseErrorFJ(Exception):
    """Excepción base para todos los errores controlados del sistema Software FJ."""
    pass  # Hereda de Exception para agrupar las excepciones del dominio


class ErrorDatoInvalidoFJ(BaseErrorFJ):
    """Se lanza cuando un dato es inválido, vacío, fuera de rango o de un tipo incorrecto."""
    pass  # Excepción específica para fallos en validaciones de entrada


class ErrorServicioOcupadoFJ(BaseErrorFJ):
    """Se lanza cuando un servicio no está disponible para ser reservado."""
    pass  # Excepción específica para problemas de disponibilidad


class ErrorEstadoReservaFJ(BaseErrorFJ):
    """Se lanza cuando se intenta una operación inválida sobre una reserva."""
    pass  # Excepción específica para reglas de confirmación o cancelación


class ErrorCalculoTarifaFJ(BaseErrorFJ):
    """Se lanza cuando hay inconsistencias al calcular costos, descuentos, cargos o impuestos."""
    pass  # Excepción específica para fallos en cálculos financieros


print("Paso 2 verificado: excepciones personalizadas listas.")  # Confirma ejecución de la celda

Paso 2 verificado: excepciones personalizadas listas.


In [3]:
class EntidadBaseFJ(ABC):
    """Clase abstracta que representa entidades generales dentro de Software FJ."""

    def __init__(self, id_registro: str) -> None:
        """Valida e inicializa el identificador único y la fecha de creación."""
        if not isinstance(id_registro, str) or len(id_registro.strip()) < 3:  # Verifica que el ID sea texto de mínimo 3 caracteres
            raise ErrorDatoInvalidoFJ("El identificador debe ser un texto de al menos 3 caracteres.")  # Lanza error si no cumple
        self._id_registro = id_registro.strip().upper()  # Guarda el ID normalizado en atributo protegido
        self._fecha_alta = datetime.now()  # Registra la fecha y hora de creación del objeto

    @property
    def id_registro(self) -> str:
        """Devuelve el identificador único de la entidad."""
        return self._id_registro  # Retorna _id_registro

    @abstractmethod
    def resumir_entidad(self) -> str:
        """Método abstracto que obliga a cada subclase a entregar su propio resumen."""
        pass  # Sobrescrito en cada clase derivada


class ClienteFJ(EntidadBaseFJ):
    """Representa a un cliente de Software FJ con atributos encapsulados y validados."""

    def __init__(self, id_registro: str, nombre: str, correo: str, telefono: str, activo: bool = True) -> None:
        """Crea un cliente validando nombre, correo, teléfono y estado inicial."""
        super().__init__(id_registro)  # Valida y asigna id_registro desde EntidadBaseFJ
        self.nombre = nombre  # Invoca el setter de nombre para validar
        self.correo = correo  # Invoca el setter de correo para validar
        self.telefono = telefono  # Invoca el setter de telefono para validar
        self._activo = bool(activo)  # Almacena estado activo/inactivo como booleano

    @property
    def nombre(self) -> str:
        """Obtiene el nombre encapsulado del cliente."""
        return self._nombre  # Devuelve _nombre

    @nombre.setter
    def nombre(self, valor: str) -> None:
        """Valida y asigna el nombre del cliente."""
        if not isinstance(valor, str) or len(valor.strip()) < 4:  # Exige texto de al menos 4 caracteres
            raise ErrorDatoInvalidoFJ("El nombre del cliente debe tener al menos 4 caracteres.")  # Lanza error de validación
        self._nombre = valor.strip()  # Guarda el nombre sin espacios extremos

    @property
    def correo(self) -> str:
        """Obtiene el correo electrónico encapsulado del cliente."""
        return self._correo  # Devuelve _correo

    @correo.setter
    def correo(self, valor: str) -> None:
        """Valida que el correo tenga '@' y dominio con punto."""
        if not isinstance(valor, str) or "@" not in valor or "." not in valor.split("@")[-1]:  # Verifica formato mínimo de correo
            raise ErrorDatoInvalidoFJ(f"Correo inválido: '{valor}'.")  # Lanza error si no cumple
        self._correo = valor.strip().lower()  # Guarda el correo en minúsculas

    @property
    def telefono(self) -> str:
        """Obtiene el teléfono encapsulado del cliente."""
        return self._telefono  # Devuelve _telefono

    @telefono.setter
    def telefono(self, valor: str) -> None:
        """Valida que el teléfono tenga entre 7 y 15 dígitos numéricos."""
        if not isinstance(valor, str) or not valor.strip().isdigit() or not (7 <= len(valor.strip()) <= 15):  # Verifica dígitos y longitud
            raise ErrorDatoInvalidoFJ(f"Teléfono inválido: '{valor}'. Debe tener entre 7 y 15 dígitos.")  # Lanza error si no cumple
        self._telefono = valor.strip()  # Guarda el teléfono limpio

    @property
    def activo(self) -> bool:
        """Indica si el cliente está habilitado para realizar reservas."""
        return self._activo  # Retorna _activo

    def bloquear_cliente(self) -> None:
        """Cambia el estado del cliente a inactivo."""
        self._activo = False  # Deshabilita al cliente para nuevas reservas

    def resumir_entidad(self) -> str:
        """Implementación polimórfica del resumen de ClienteFJ."""
        estado_txt = "Activo" if self._activo else "Inactivo"  # Traduce booleano a etiqueta legible
        return f"Cliente[{self._id_registro}] {self._nombre} | {self._correo} | Tel: {self._telefono} | {estado_txt}"  # Retorna resumen


print("Paso 3 verificado: EntidadBaseFJ y ClienteFJ cargados.")  # Confirma ejecución de la celda

Paso 3 verificado: EntidadBaseFJ y ClienteFJ cargados.


In [5]:
class ServicioBaseFJ(EntidadBaseFJ, ABC):
    """Clase abstracta que define atributos y comportamientos comunes de los servicios de Software FJ."""

    def __init__(self, id_registro: str, nombre_servicio: str, tarifa_unitaria: float, disponible: bool = True) -> None:
        """Inicializa atributos comunes del servicio y valida la tarifa unitaria."""
        super().__init__(id_registro)  # Valida y asigna id_registro desde EntidadBaseFJ
        if not isinstance(nombre_servicio, str) or not nombre_servicio.strip():  # Verifica que el nombre no esté vacío
            raise ErrorDatoInvalidoFJ("El nombre del servicio es obligatorio.")  # Lanza error si falta el nombre

        try:
            tarifa_num = float(tarifa_unitaria)  # Convierte la tarifa recibida a número decimal float
        except (TypeError, ValueError) as exc_tarifa:
            raise ErrorDatoInvalidoFJ("La tarifa unitaria debe ser un valor numérico válido.") from exc_tarifa  # Encadena excepción

        if tarifa_num <= 0:  # Verifica que la tarifa sea positiva
            raise ErrorDatoInvalidoFJ("La tarifa unitaria debe ser mayor que cero.")  # Rechaza cero o negativos

        self._nombre_servicio = nombre_servicio.strip()  # Guarda nombre del servicio
        self._tarifa_unitaria = tarifa_num  # Guarda tarifa validada
        self._disponible = bool(disponible)  # Guarda disponibilidad inicial

    @property
    def nombre_servicio(self) -> str:
        """Devuelve el nombre del servicio."""
        return self._nombre_servicio  # Retorna _nombre_servicio

    @property
    def disponible(self) -> bool:
        """Indica si el servicio está disponible para reservar."""
        return self._disponible  # Retorna _disponible

    def actualizar_disponibilidad(self, estado: bool) -> None:
        """Actualiza la disponibilidad del servicio."""
        self._disponible = bool(estado)  # Asigna nuevo estado booleano

    @abstractmethod
    def validar_cantidad(self, cantidad: int) -> None:
        """Valida si la cantidad de horas o días es válida para el servicio."""
        pass  # Sobrescrito en cada subclase

    @abstractmethod
    def calcular_base(self, cantidad: int) -> float:
        """Calcula el subtotal base antes de descuentos, cargos o impuestos."""
        pass  # Sobrescrito en cada subclase

    @abstractmethod
    def describir_servicio(self) -> str:
        """Devuelve la descripción específica del servicio."""
        pass  # Sobrescrito en cada subclase

    def calcular_costo(self, cantidad: int, descuento: float = 0.0, iva: float = 0.19, recargo: float = 0.0) -> float:
        """
        Calcula el costo total usando parámetros opcionales (variantes de cálculo con
        descuento, IVA y recargo) previa validación de unidades y coherencia financiera.
        """
        self.validar_cantidad(cantidad)  # Valida la cantidad según las reglas de cada subclase

        try:
            subtotal = self.calcular_base(cantidad)  # Calcula el subtotal aplicando polimorfismo
            desc_val = float(descuento)  # Convierte descuento a float
            iva_val = float(iva)  # Convierte IVA a float
            recargo_val = float(recargo)  # Convierte recargo a float
        except (TypeError, ValueError) as exc_calc:
            raise ErrorCalculoTarifaFJ("Los parámetros del cálculo deben ser numéricos.") from exc_calc  # Encadena excepción

        if not (0.0 <= desc_val < 1.0):  # Valida que el descuento esté entre 0.0 y 0.99
            raise ErrorCalculoTarifaFJ("El descuento debe estar en el rango [0.0, 1.0).")  # Lanza error si es inválido
        if iva_val < 0.0 or recargo_val < 0.0:  # Evita IVA o recargo negativos
            raise ErrorCalculoTarifaFJ("El IVA y el recargo no pueden ser negativos.")  # Lanza error si son negativos

        total = (subtotal * (1.0 - desc_val) + recargo_val) * (1.0 + iva_val)  # Aplica descuento, recargo e IVA
        return round(total, 2)  # Redondea el resultado a dos decimales

    def resumir_entidad(self) -> str:
        """Implementación polimórfica del resumen de ServicioBaseFJ."""
        disp_txt = "Disponible" if self._disponible else "No disponible"  # Define texto de disponibilidad
        return f"{self.describir_servicio()} | Estado: {disp_txt}"  # Retorna resumen completo


class SalaTrabajoFJ(ServicioBaseFJ):
    """Servicio especializado para reserva de salas por horas."""

    def __init__(self, id_registro: str, nombre_servicio: str, tarifa_unitaria: float, capacidad: int, disponible: bool = True) -> None:
        """Inicializa la sala y valida su capacidad máxima."""
        super().__init__(id_registro, nombre_servicio, tarifa_unitaria, disponible)  # Inicializa atributos base
        if not isinstance(capacidad, int) or isinstance(capacidad, bool) or capacidad < 2:  # Valida entero >= 2
            raise ErrorDatoInvalidoFJ("La capacidad de la sala debe ser un entero de al menos 2 personas.")  # Lanza error
        self._capacidad = capacidad  # Guarda capacidad

    def validar_cantidad(self, cantidad: int) -> None:
        """Valida que la reserva de sala sea de 1 a 10 horas enteras."""
        if not isinstance(cantidad, int) or isinstance(cantidad, bool) or not (1 <= cantidad <= 10):  # Valida rango [1, 10]
            raise ErrorDatoInvalidoFJ("La sala admite reservas entre 1 y 10 horas enteras.")  # Lanza error fuera de rango

    def calcular_base(self, cantidad: int) -> float:
        """Calcula el subtotal por horas reservadas."""
        return self._tarifa_unitaria * cantidad  # Multiplica tarifa por horas

    def describir_servicio(self) -> str:
        """Describe la sala y su capacidad."""
        return f"Sala[{self._id_registro}] '{self._nombre_servicio}' (Cap. {self._capacidad} pers. | ${self._tarifa_unitaria:,.0f}/h)"  # Retorna detalle


class EquipoAlquilerFJ(ServicioBaseFJ):
    """Servicio especializado para alquiler de equipos por días."""

    def __init__(self, id_registro: str, nombre_servicio: str, tarifa_unitaria: float, codigo_activo: str, disponible: bool = True) -> None:
        """Inicializa el equipo y valida su código de activo."""
        super().__init__(id_registro, nombre_servicio, tarifa_unitaria, disponible)  # Inicializa atributos base
        if not isinstance(codigo_activo, str) or len(codigo_activo.strip()) < 4:  # Exige código de al menos 4 caracteres
            raise ErrorDatoInvalidoFJ("El código de activo del equipo debe tener al menos 4 caracteres.")  # Lanza error
        self._codigo_activo = codigo_activo.strip().upper()  # Guarda código normalizado

    def validar_cantidad(self, cantidad: int) -> None:
        """Valida que el alquiler de equipo sea de 1 a 30 días enteros."""
        if not isinstance(cantidad, int) or isinstance(cantidad, bool) or not (1 <= cantidad <= 30):  # Valida rango [1, 30]
            raise ErrorDatoInvalidoFJ("El alquiler de equipos admite entre 1 y 30 días enteros.")  # Lanza error fuera de rango

    def calcular_base(self, cantidad: int) -> float:
        """Calcula el subtotal por días con 5% de soporte si supera 5 días."""
        base = self._tarifa_unitaria * cantidad  # Calcula costo base diario
        return base * 1.05 if cantidad > 5 else base  # Aplica recargo de soporte en alquileres mayores a 5 días

    def describir_servicio(self) -> str:
        """Describe el equipo y su código de activo."""
        return f"Equipo[{self._id_registro}] '{self._nombre_servicio}' (Activo: {self._codigo_activo} | ${self._tarifa_unitaria:,.0f}/día)"  # Retorna detalle


class AsesoriaIngenieriaFJ(ServicioBaseFJ):
    """Servicio especializado para asesorías técnicas por horas."""

    def __init__(self, id_registro: str, nombre_servicio: str, tarifa_unitaria: float, linea_tecnica: str, disponible: bool = True) -> None:
        """Inicializa la asesoría y valida la línea técnica."""
        super().__init__(id_registro, nombre_servicio, tarifa_unitaria, disponible)  # Inicializa atributos base
        if not isinstance(linea_tecnica, str) or len(linea_tecnica.strip()) < 3:  # Exige línea técnica de al menos 3 caracteres
            raise ErrorDatoInvalidoFJ("La línea técnica de la asesoría debe tener al menos 3 caracteres.")  # Lanza error
        self._linea_tecnica = linea_tecnica.strip()  # Guarda línea técnica

    def validar_cantidad(self, cantidad: int) -> None:
        """Valida que la asesoría sea de 2 a 40 horas enteras."""
        if not isinstance(cantidad, int) or isinstance(cantidad, bool) or not (2 <= cantidad <= 40):  # Valida rango [2, 40]
            raise ErrorDatoInvalidoFJ("La asesoría técnica admite entre 2 y 40 horas enteras.")  # Lanza error fuera de rango

    def calcular_base(self, cantidad: int) -> float:
        """Calcula el subtotal por horas de asesoría."""
        return self._tarifa_unitaria * cantidad  # Multiplica tarifa por horas

    def describir_servicio(self) -> str:
        """Describe la asesoría y su línea técnica."""
        return f"Asesoría[{self._id_registro}] '{self._nombre_servicio}' (Línea: {self._linea_tecnica} | ${self._tarifa_unitaria:,.0f}/h)"  # Retorna detalle


print("Paso 4 verificado: ServicioBaseFJ y 3 servicios derivados cargados.")  # Confirma ejecución de la celda

Paso 4 verificado: ServicioBaseFJ y 3 servicios derivados cargados.


In [6]:
class ReservaFJ(EntidadBaseFJ):
    """Integra cliente, servicio, duración, costo y estado con confirmación, cancelación y procesamiento."""

    def __init__(
        self,
        id_registro: str,
        cliente: ClienteFJ,
        servicio: ServicioBaseFJ,
        duracion: int,
        descuento: float = 0.0,
        iva: float = 0.19,
        recargo: float = 0.0
    ) -> None:
        """Crea y valida una reserva verificando cliente, servicio y cálculo de costo."""
        super().__init__(id_registro)  # Valida y asigna id_registro

        if not isinstance(cliente, ClienteFJ):  # Verifica que cliente sea instancia de ClienteFJ
            raise ErrorDatoInvalidoFJ("Debe suministrar un objeto ClienteFJ válido.")  # Lanza error si no lo es
        if not cliente.activo:  # Verifica que el cliente esté activo
            raise ErrorEstadoReservaFJ(f"El cliente '{cliente.id_registro}' está inactivo y no puede reservar.")  # Bloquea reserva
        if not isinstance(servicio, ServicioBaseFJ):  # Verifica que servicio sea instancia de ServicioBaseFJ
            raise ErrorDatoInvalidoFJ("Debe suministrar un objeto ServicioBaseFJ válido.")  # Lanza error si no lo es
        if not servicio.disponible:  # Verifica que el servicio esté disponible
            raise ErrorServicioOcupadoFJ(f"El servicio '{servicio.id_registro}' no está disponible.")  # Lanza error de disponibilidad

        try:
            # Calcula el costo final delegando a calcular_costo del servicio
            self._costo_final = servicio.calcular_costo(
                cantidad=duracion,
                descuento=descuento,
                iva=iva,
                recargo=recargo
            )
        except BaseErrorFJ as exc_costo:
            # Encadena la excepción original dentro de un error de reserva
            raise ErrorEstadoReservaFJ(f"No se pudo calcular el costo de la reserva '{id_registro}'.") from exc_costo

        self._cliente = cliente  # Asigna cliente validado
        self._servicio = servicio  # Asigna servicio validado
        self._duracion = duracion  # Asigna duración
        self._estado = "PENDIENTE"  # Inicializa estado en PENDIENTE

    @property
    def estado(self) -> str:
        """Devuelve el estado actual de la reserva."""
        return self._estado  # Retorna _estado

    @property
    def costo_final(self) -> float:
        """Devuelve el costo final calculado de la reserva."""
        return self._costo_final  # Retorna _costo_final

    def confirmar_reserva(self) -> None:
        """Confirma la reserva si está pendiente y marca el servicio como no disponible."""
        if self._estado == "CANCELADA":  # Impide confirmar una reserva cancelada
            raise ErrorEstadoReservaFJ(f"La reserva '{self._id_registro}' está CANCELADA y no puede confirmarse.")  # Lanza error
        if self._estado == "CONFIRMADA":  # Impide confirmar una reserva ya confirmada
            raise ErrorEstadoReservaFJ(f"La reserva '{self._id_registro}' ya está CONFIRMADA.")  # Lanza error
        self._estado = "CONFIRMADA"  # Actualiza estado a CONFIRMADA
        self._servicio.actualizar_disponibilidad(False)  # Marca el servicio como ocupado

    def cancelar_reserva(self) -> None:
        """Cancela la reserva si no está cancelada previamente y libera el servicio."""
        if self._estado == "CANCELADA":  # Evita cancelar dos veces la misma reserva
            raise ErrorEstadoReservaFJ(f"La reserva '{self._id_registro}' ya se encuentra CANCELADA.")  # Lanza error
        self._estado = "CANCELADA"  # Actualiza estado a CANCELADA
        self._servicio.actualizar_disponibilidad(True)  # Libera el servicio para futuras reservas

    def procesar(self, accion: str) -> str:
        """Procesa una acción ('confirmar' o 'cancelar') sobre la reserva con manejo de errores."""
        if not isinstance(accion, str) or not accion.strip():  # Verifica que la acción sea texto no vacío
            raise ErrorEstadoReservaFJ("La acción debe ser un texto ('confirmar' o 'cancelar').")  # Lanza error si es inválida

        accion_limpia = accion.strip().lower()  # Normaliza texto a minúsculas
        if accion_limpia == "confirmar":  # Ejecuta confirmación
            self.confirmar_reserva()  # Llama confirmar_reserva
            return f"Reserva {self._id_registro} CONFIRMADA por ${self._costo_final:,.2f}."  # Retorna mensaje de éxito
        if accion_limpia == "cancelar":  # Ejecuta cancelación
            self.cancelar_reserva()  # Llama cancelar_reserva
            return f"Reserva {self._id_registro} CANCELADA; servicio '{self._servicio.id_registro}' disponible."  # Retorna mensaje

        raise ErrorEstadoReservaFJ(f"Acción no válida para reserva: '{accion}'.")  # Rechaza acciones desconocidas

    def resumir_entidad(self) -> str:
        """Implementación polimórfica del resumen de ReservaFJ."""
        return (
            f"Reserva[{self._id_registro}] | Cliente: {self._cliente.id_registro} | "
            f"Servicio: {self._servicio.id_registro} | Duración: {self._duracion} | "
            f"Total: ${self._costo_final:,.2f} | Estado: {self._estado}"
        )  # Retorna resumen completo


class GestorReservasFJ:
    """Gestiona las listas internas en memoria de clientes, servicios y reservas sin base de datos."""

    def __init__(self) -> None:
        """Inicializa las listas internas del sistema."""
        self.clientes: list[ClienteFJ] = []  # Lista interna de clientes
        self.servicios: list[ServicioBaseFJ] = []  # Lista interna de servicios
        self.reservas: list[ReservaFJ] = []  # Lista interna de reservas

    def registrar_cliente(self, cliente: ClienteFJ) -> str:
        """Agrega un cliente a la lista interna verificando que no exista duplicado."""
        if any(c.id_registro == cliente.id_registro for c in self.clientes):  # Busca ID repetido
            raise ErrorDatoInvalidoFJ(f"Ya existe un cliente con ID '{cliente.id_registro}'.")  # Impide duplicados
        self.clientes.append(cliente)  # Agrega cliente a la lista
        return f"Cliente agregado -> {cliente.resumir_entidad()}"  # Retorna confirmación

    def registrar_servicio(self, servicio: ServicioBaseFJ) -> str:
        """Agrega un servicio a la lista interna verificando que no exista duplicado."""
        if any(s.id_registro == servicio.id_registro for s in self.servicios):  # Busca ID repetido
            raise ErrorDatoInvalidoFJ(f"Ya existe un servicio con ID '{servicio.id_registro}'.")  # Impide duplicados
        self.servicios.append(servicio)  # Agrega servicio a la lista
        return f"Servicio agregado -> {servicio.resumir_entidad()}"  # Retorna confirmación

    def registrar_reserva(self, reserva: ReservaFJ) -> str:
        """Agrega una reserva a la lista interna verificando que no exista duplicado."""
        if any(r.id_registro == reserva.id_registro for r in self.reservas):  # Busca ID repetido
            raise ErrorDatoInvalidoFJ(f"Ya existe una reserva con ID '{reserva.id_registro}'.")  # Impide duplicados
        self.reservas.append(reserva)  # Agrega reserva a la lista
        return f"Reserva registrada -> {reserva.resumir_entidad()}"  # Retorna confirmación


print("Paso 5 verificado: ReservaFJ y GestorReservasFJ cargados.")  # Confirma ejecución de la celda

Paso 5 verificado: ReservaFJ y GestorReservasFJ cargados.


In [7]:
def ejecutar_operacion_segura(idx: int, etiqueta: str, fn_operacion) -> dict:
    """
    Ejecuta una operación usando try / except / else / finally, captura excepciones
    propias y encadenadas, registra en la bitácora y devuelve un resumen para auditoría.
    """
    print(f"\n--- Operación {idx:02d}: {etiqueta} ---")  # Muestra encabezado de la operación
    estado_cierre = "PENDIENTE"  # Estado inicial para el bloque finally
    detalle_salida = ""  # Texto descriptivo del resultado

    try:
        resultado = fn_operacion()  # Ejecuta la operación recibida
    except BaseErrorFJ as err_propio:
        # Si existe una excepción encadenada con 'from', muestra su tipo y mensaje
        origen = f" | Origen encadenado: {type(err_propio.__cause__).__name__} ({err_propio.__cause__})" if err_propio.__cause__ else ""
        msg_error = f"OP-{idx:02d} [{etiqueta}] FALLÓ -> {type(err_propio).__name__}: {err_propio}{origen}"  # Construye mensaje
        bitacora.error(msg_error)  # Registra el error en bitacora_software_fj.log
        print(f"  [ERROR CONTROLADO] {msg_error}")  # Muestra en consola sin detener el programa
        estado_cierre = "ERROR_CONTROLADO"  # Actualiza estado para finally
        detalle_salida = f"{type(err_propio).__name__}: {err_propio}"  # Guarda detalle para tabla
    except Exception as err_general:
        # Captura cualquier otra excepción para mantener la estabilidad del sistema
        msg_error = f"OP-{idx:02d} [{etiqueta}] ERROR GENERAL -> {type(err_general).__name__}: {err_general}"  # Construye mensaje
        bitacora.error(msg_error)  # Registra error en bitacora_software_fj.log
        print(f"  [ERROR GENERAL] {msg_error}")  # Muestra en consola
        estado_cierre = "ERROR_GENERAL"  # Actualiza estado para finally
        detalle_salida = f"{type(err_general).__name__}: {err_general}"  # Guarda detalle para tabla
    else:
        # Se ejecuta solo cuando try finaliza sin lanzar excepciones
        msg_ok = f"OP-{idx:02d} [{etiqueta}] EXITOSA -> {resultado}"  # Construye mensaje de éxito
        bitacora.info(msg_ok)  # Registra el evento exitoso en bitacora_software_fj.log
        print(f"  [ÉXITO] {resultado}")  # Muestra resultado en consola
        estado_cierre = "EXITOSA"  # Actualiza estado a EXITOSA
        detalle_salida = str(resultado)  # Guarda detalle para tabla
    finally:
        # Se ejecuta siempre al finalizar cada operación, con o sin error
        bitacora.info(f"Cierre OP-{idx:02d} | Estado final: {estado_cierre}")  # Registra cierre en bitácora

    return {
        "Operación": f"OP-{idx:02d}",
        "Escenario": etiqueta,
        "Estado": estado_cierre,
        "Detalle": detalle_salida
    }  # Devuelve registro estructurado de la operación


def correr_simulacion_fj() -> tuple[GestorReservasFJ, list[dict]]:
    """Ejecuta 12 operaciones completas con casos válidos e inválidos y retorna gestor y reporte."""
    gestor = GestorReservasFJ()  # Crea instancia del gestor en memoria
    objetos_prueba: dict = {}  # Diccionario para compartir objetos creados entre pasos
    reporte_ops: list[dict] = []  # Lista de resúmenes para verificación tabular
    bitacora.info("=== INICIO DE SIMULACIÓN FASE 4 - SOFTWARE FJ ===")  # Registra inicio en bitácora

    # 1. Registro válido de cliente 1
    def op_01():
        cli_a = ClienteFJ("CLI-01", "Camila Vargas", "camila.vargas@empresa.com", "3112233445", activo=True)  # Crea cliente válido
        objetos_prueba["cli_a"] = cli_a  # Guarda referencia
        return gestor.registrar_cliente(cli_a)  # Registra en lista interna

    # 2. Registro inválido de cliente (correo sin arroba)
    def op_02():
        cli_err = ClienteFJ("CLI-02", "Daniel Soto", "daniel_correo_malo.com", "3123344556")  # Falla validación de correo
        return gestor.registrar_cliente(cli_err)  # No se alcanza

    # 3. Registro válido de clientes 2 y 3 (el cliente 3 se bloquea para probar restricción)
    def op_03():
        cli_b = ClienteFJ("CLI-03", "Natalia Rojas", "natalia.rojas@estudio.co", "3134455667", activo=True)  # Crea cliente válido
        cli_c = ClienteFJ("CLI-04", "Andrés Peña", "andres.pena@servicios.org", "3145566778", activo=True)  # Crea cliente válido
        cli_c.bloquear_cliente()  # Bloquea al cliente 3
        objetos_prueba["cli_b"] = cli_b  # Guarda referencia
        objetos_prueba["cli_c"] = cli_c  # Guarda referencia
        return f"{gestor.registrar_cliente(cli_b)} / {gestor.registrar_cliente(cli_c)}"  # Registra ambos

    # 4. Creación válida de servicio 1: SalaTrabajoFJ
    def op_04():
        sala_1 = SalaTrabajoFJ("SRV-01", "Sala Reuniones Central", tarifa_unitaria=75000.0, capacidad=10, disponible=True)  # Crea sala válida
        objetos_prueba["sala_1"] = sala_1  # Guarda referencia
        return gestor.registrar_servicio(sala_1)  # Registra servicio

    # 5. Creación válida de servicio 2: EquipoAlquilerFJ
    def op_05():
        eq_1 = EquipoAlquilerFJ("SRV-02", "Laptop Desarrollo 16GB", tarifa_unitaria=48000.0, codigo_activo="ACT-NB-012", disponible=True)  # Crea equipo válido
        objetos_prueba["eq_1"] = eq_1  # Guarda referencia
        return gestor.registrar_servicio(eq_1)  # Registra servicio

    # 6. Creación válida de servicio 3: AsesoriaIngenieriaFJ
    def op_06():
        ase_1 = AsesoriaIngenieriaFJ("SRV-03", "Revisión Código y POO", tarifa_unitaria=110000.0, linea_tecnica="Arquitectura Software", disponible=True)  # Crea asesoría válida
        objetos_prueba["ase_1"] = ase_1  # Guarda referencia
        return gestor.registrar_servicio(ase_1)  # Registra servicio

    # 7. Creación inválida de servicio (tarifa no numérica -> encadena excepción)
    def op_07():
        sala_err = SalaTrabajoFJ("SRV-04", "Sala Alterna", tarifa_unitaria="no_es_numero", capacidad=6)  # Encadena ValueError en ErrorDatoInvalidoFJ
        return gestor.registrar_servicio(sala_err)  # No se alcanza

    # 8. Reserva exitosa con descuento del 15% e IVA del 19% (parámetros opcionales)
    def op_08():
        res_1 = ReservaFJ("RES-01", objetos_prueba["cli_a"], objetos_prueba["sala_1"], duracion=3, descuento=0.15, iva=0.19)  # Crea reserva válida
        objetos_prueba["res_1"] = res_1  # Guarda referencia
        alta = gestor.registrar_reserva(res_1)  # Registra en lista
        conf = res_1.procesar("confirmar")  # Confirma y ocupa la sala
        return f"{alta} -> {conf}"  # Retorna resultado

    # 9. Reserva fallida por servicio ocupado (sala_1 ya ocupada en OP-08)
    def op_09():
        res_err_disp = ReservaFJ("RES-02", objetos_prueba["cli_b"], objetos_prueba["sala_1"], duracion=2)  # Lanza ErrorServicioOcupadoFJ
        return gestor.registrar_reserva(res_err_disp)  # No se alcanza

    # 10. Reserva fallida por cliente inactivo/bloqueado
    def op_10():
        res_err_cli = ReservaFJ("RES-03", objetos_prueba["cli_c"], objetos_prueba["eq_1"], duracion=4)  # Lanza ErrorEstadoReservaFJ
        return gestor.registrar_reserva(res_err_cli)  # No se alcanza

    # 11. Reserva fallida por duración excedida (encadenamiento de excepciones)
    def op_11():
        res_err_dur = ReservaFJ("RES-04", objetos_prueba["cli_b"], objetos_prueba["ase_1"], duracion=55)  # 55h excede máx 40h -> encadena excepción
        return gestor.registrar_reserva(res_err_dur)  # No se alcanza

    # 12. Reserva exitosa de equipo (6 días con recargo), confirmación y cancelación
    def op_12():
        res_2 = ReservaFJ("RES-05", objetos_prueba["cli_b"], objetos_prueba["eq_1"], duracion=6, recargo=10000.0)  # Crea reserva válida con recargo
        gestor.registrar_reserva(res_2)  # Registra reserva
        res_2.procesar("confirmar")  # Confirma reserva
        canc = res_2.procesar("cancelar")  # Cancela reserva y libera equipo
        resumen_listas = f"Clientes={len(gestor.clientes)}, Servicios={len(gestor.servicios)}, Reservas={len(gestor.reservas)}"  # Conteo final
        return f"{canc} | {resumen_listas}"  # Retorna resumen

    operaciones = [
        (1, "Registro válido de cliente principal", op_01),
        (2, "Registro inválido de cliente (correo)", op_02),
        (3, "Registro de clientes adicionales (uno bloqueado)", op_03),
        (4, "Creación válida de SalaTrabajoFJ", op_04),
        (5, "Creación válida de EquipoAlquilerFJ", op_05),
        (6, "Creación válida de AsesoriaIngenieriaFJ", op_06),
        (7, "Creación inválida de servicio (tarifa no numérica)", op_07),
        (8, "Reserva y confirmación de sala con descuento/IVA", op_08),
        (9, "Reserva fallida por servicio no disponible", op_09),
        (10, "Reserva fallida por cliente inactivo", op_10),
        (11, "Reserva fallida por horas fuera de rango", op_11),
        (12, "Reserva de equipo, confirmación y cancelación", op_12),
    ]

    for num_op, desc_op, fn_op in operaciones:
        reporte_ops.append(ejecutar_operacion_segura(num_op, desc_op, fn_op))  # Ejecuta cada operación en orden

    bitacora.info("=== FIN DE SIMULACIÓN FASE 4 - SOFTWARE FJ ===")  # Registra cierre en bitácora
    return gestor, reporte_ops  # Retorna gestor y lista de resultados


gestor_final, reporte_simulacion = correr_simulacion_fj()  # Ejecuta la simulación completa


--- Operación 01: Registro válido de cliente principal ---
  [ÉXITO] Cliente agregado -> Cliente[CLI-01] Camila Vargas | camila.vargas@empresa.com | Tel: 3112233445 | Activo

--- Operación 02: Registro inválido de cliente (correo) ---
  [ERROR CONTROLADO] OP-02 [Registro inválido de cliente (correo)] FALLÓ -> ErrorDatoInvalidoFJ: Correo inválido: 'daniel_correo_malo.com'.

--- Operación 03: Registro de clientes adicionales (uno bloqueado) ---
  [ÉXITO] Cliente agregado -> Cliente[CLI-03] Natalia Rojas | natalia.rojas@estudio.co | Tel: 3134455667 | Activo / Cliente agregado -> Cliente[CLI-04] Andrés Peña | andres.pena@servicios.org | Tel: 3145566778 | Inactivo

--- Operación 04: Creación válida de SalaTrabajoFJ ---
  [ÉXITO] Servicio agregado -> Sala[SRV-01] 'Sala Reuniones Central' (Cap. 10 pers. | $75,000/h) | Estado: Disponible

--- Operación 05: Creación válida de EquipoAlquilerFJ ---
  [ÉXITO] Servicio agregado -> Equipo[SRV-02] 'Laptop Desarrollo 16GB' (Activo: ACT-NB-012 | $48,0

In [8]:
# Muestra una tabla clara para verificar el resultado de cada operación ejecutada
print(f"{'OP':<7} | {'ESTADO':<18} | {'ESCENARIO'}")  # Cabecera de la tabla de verificación
print("-" * 80)  # Línea separadora
for item in reporte_simulacion:
    print(f"{item['Operación']:<7} | {item['Estado']:<18} | {item['Escenario']}")  # Imprime fila por operación

print("\n=== CONTENIDO DEL ARCHIVO DE LOGS (bitacora_software_fj.log) ===\n")  # Encabezado del archivo de logs
try:
    with open("bitacora_software_fj.log", mode="r", encoding="utf-8") as f_log:  # Abre el archivo en modo lectura
        print(f_log.read())  # Imprime todas las líneas registradas en la bitácora
except FileNotFoundError as err_log:
    print(f"No se encontró el archivo de logs: {err_log}")  # Controla si no se ha corrido la Celda 6

OP      | ESTADO             | ESCENARIO
--------------------------------------------------------------------------------
OP-01   | EXITOSA            | Registro válido de cliente principal
OP-02   | ERROR_CONTROLADO   | Registro inválido de cliente (correo)
OP-03   | EXITOSA            | Registro de clientes adicionales (uno bloqueado)
OP-04   | EXITOSA            | Creación válida de SalaTrabajoFJ
OP-05   | EXITOSA            | Creación válida de EquipoAlquilerFJ
OP-06   | EXITOSA            | Creación válida de AsesoriaIngenieriaFJ
OP-07   | ERROR_CONTROLADO   | Creación inválida de servicio (tarifa no numérica)
OP-08   | EXITOSA            | Reserva y confirmación de sala con descuento/IVA
OP-09   | ERROR_CONTROLADO   | Reserva fallida por servicio no disponible
OP-10   | ERROR_CONTROLADO   | Reserva fallida por cliente inactivo
OP-11   | ERROR_CONTROLADO   | Reserva fallida por horas fuera de rango
OP-12   | EXITOSA            | Reserva de equipo, confirmación y cancelación

=== C